# Programming exercise 5: Collective Ising spins

Due on Monday, 18.05.2026, 20.00h

## The problem

The collective Ising spin-model with a transverse field is governed by the Hamiltonian
$$
H=-\frac{J}{N}S_z^2 - \Omega S_x
$$
with the collective spin operators $S_\alpha = \sum_i \sigma_i^\alpha/2$, where $\sigma_i^\alpha$ is a Pauli operator acting on spin $i$. This model features a quantum phase transition at $\Omega/J=1$, meaning that the ground-state properties change abruptly. It also shows a dynamical phase transition at $\Omega/J=0.5$, meaning that the long-time properties of the dynamics change drastically at this point. The dynamical case was recently studied experimentally using a chain of trapped ions (https://www.nature.com/articles/nature24654). Note the different convention (x↔z) and the different definitions of field and interaction strength used there. In the experiment, the interactions were not strictly infinite-range but followed a power-law with a small exponent; the phenomenology remains the same.

We want to use this model to see how phase transitions can be observed numerically by extrapolating to infinite $N$, and to see how symmetries can be exploited to greatly reduce the size of the relevant Hilbert space of a spin system.

In [2]:
# load standard libraries

import numpy as np   # standard numerics library
import numpy.linalg as LA
import scipy.linalg as sLA

import matplotlib.pyplot as plt   # for making plots

import time as time

import scipy.sparse as sparse
import scipy.sparse.linalg as sLA

import Comp_Quant_Dynam as cqd

### Exercise 1

We first want to study the low-lying eigenstates and observe the quantum phase transition. Thus, it makes sense to work with sparse matrices.

Generate the Hamiltonian as a sparse matrix in the Dicke state basis. You can proceed similarly to the coupled oscillator problem from Programming exercise 4: Build the matrix representation of the *collective* spin operator $S_+$ and $S_z$, and then use these to construct any other operators you need, e.g. $S_x=(S_+ + S_-)/2$. For matrix-matrix and matrix-vector multiplication it is intuitive to use the @ operator, which is a short hand for the dot() function.

It is crucial that you use the Dicke state basis as discussed in the lecture. That way you only need N+1 basis states, which is a huge reduction of complexity compared to using the product Hilbert space of N spin 1/2 particles, which has size $2^N$.

Test your implementation for N=20 by using small (zero) and large values of $\Omega$ and comparing (qualitatively) to the expectation of $S_z^2$ and $S_x$ in the ground state in either limit. Set J=1 from now on, meaning that times are in units of 1/J and when we speak of $\Omega$ we really mean the dimensionless parameter $\Omega/J$.

### Exercise 2

Loop over $\Omega$ (from 0 to 2 in small steps) using at least $N=100$ and record the lowest 3 eigenenergies and, for the ground state, the probabilities P(M) to be in state $|N/2,M\rangle$. 
Plot P(M;$\Omega$) in a density plot (e.g. using the function plt.imshow()). At small $\Omega$ you should observe that the ground state contains only components with large z-magnetization while at large $\Omega$ the distribution is peaked around $M=0$. In between, the quantum phase transition happens. 

For small $\Omega$ one observes a kind of "spontaneous symmetry breaking." The gap between the ground and first excited state becomes very small. The states $|00...0\rangle = |N/2,N/2\rangle$ and $|11...1\rangle = |N/2,-N/2\rangle$ are degenerate ground states at $\Omega=0$. The Hamiltonian has a $Z_2$ symmetry: it is invariant under flipping all spins. Thus the eigenstates should also respect this symmetry and be either symmetric or anti-symmetric with respect to spin flips. However, when the gap between the symmetric and anti-symmetric ground states becomes too small, the numerical diagonalization algorithm may randomly choose a superposition of the two states, which can be $|N/2,-N/2\rangle$ or $|N/2,N/2\rangle$, thus breaking the symmetry.

Optional: 

We now want to analyze the gap between the ground and excited states. At a quantum phase transition, the gap closes (vanishes) in the thermodynamic limit ($N\rightarrow\infty$). The spin-flip symmetry means that, if written in the symmetrized basis, the Hamiltonian will consist of two disconnected blocks, representing the symmetric ($\propto |N/2,M\rangle + |N/2,-M\rangle$) and anti-symmetric states ($\propto |N/2,M\rangle - |N/2,-M\rangle$).

Plot the gap between the ground and first excited ($\delta_{0,1}=E_1-E_0$) and ground and second excited state ($\delta_{0,2}$). You should see that the gap $\delta_{0,1}$ becomes very small in the ferromagnetic phase ($\Omega<1$). This is the gap between the symmetric and anti-symmetric ground states. The relevant gap for the phase transition is $\delta_{0,2}$, which is the gap between the ground and first excited state within the symmetric block. This should show a minimum near the phase transition point.

One could also fix the symmetry problem by considering the symmetric and anti-symmetric blocks separately. For this you need to implement the Hamiltonian again in the symmetrized basis (or use a basis transformation into the symmetrized basis). This will give a speedup of at least a factor of 2.

### Exercise 3 (optional)

We now want to see how the gap $\delta_{0,2} = E_2-E_0$ at the critical point closes at large $N$ ("finite size scaling analysis"). For this, determine the position and size of the smallest gap as a function of N. Loop N from 40 to 400 in steps of 20. The easiest way to get the minimal gap is to calculate it on an $\Omega$-grid around 1, where the transition is expected to be. For example, increase $\Omega$ from 0.8 to 1 in 100 steps. You could also make the search for the minimal gap more efficient by implementing an iterative search procedure.

Plot the resulting distance of the optimal $\Omega$ from 1 and size of the minimal gap double logarithmically. You should get a power law behavior. Extract the exponent of this power law for both the minimal gap and its distance from 1.

### Exercise 4

Now calculate the time evolution under the above Hamiltonian, starting in the state $|11...1\rangle = |N/2,-N/2\rangle$. The most straightforward way to do this is to use scipy's expm function which allows you to simply exponentiate a matrix so you can directly calculate $\exp(-iHt)$ and apply it to the initial state (e.g. using the @ operator). Try this out, keeping in mind that `expm` expects a dense matrix as input (i.e. if you worked with sparse matrices in the previous exercises you may have to convert `H` to a dense matrix). You will see that it becomes really slow as you go to larger atom numbers. The more efficient way is to use exact diagonalization. As we need all eigenstates for this, the use of sparse matrices does not give an advantage in this case. Also, the spin-flip symmetry is now not so useful since our initial state already breaks this symmetry.

To calculate $|\psi(t)\rangle$, project the initial state onto the eigenstates. Remember that the eigenstates are in the columns of the eigenvector matrix returned by `eig` (or `eigh`). Thus the transformation into the energy eigenbasis can be achieved by applying the conjugate transpose of this matrix to the initial state. Evolve the state in the eigenbasis and transform back.

Calculate the expectation values of $S_z$ and $S_z^2$ as a function of time for some value of $\Omega$ below and above 0.5. You should see different behavior in the two regimes. Check whether your results agree qualitatively with the observations in https://www.nature.com/articles/nature24654, in particular Fig. 2 (the paper is also on arXiv: https://arxiv.org/abs/1708.01044). Note that their definition of the parameter $\Omega/J$ differs by a factor 1/2 from ours such that the transition point is at 1 for them.

### Exercise 5 (optional)

Study the long-time limit systematically as a function of $\Omega$. Loop $\Omega$ from 0 to 1 in at least 50 steps. Use at least N=100. Calculate the values of $S_z$ and $S_z^2$ at some long time. Also calculate the infinite-time average (diagonal-ensemble value)
$$
\langle O \rangle_\infty = \sum_k |\langle \psi_0|\phi_k\rangle|^2 \langle \phi_k|O|\phi_k\rangle
$$
where $|\phi_k\rangle$ are the eigenstates and $O$ is the observable of interest (here $S_z$ or $S_z^2$).
Compare your results to Fig. 6 in https://arxiv.org/abs/1708.01044. This shows the dynamical phase transition.